**Fine Tuning a SLM Model for Mental Health Awareness Bot**

In [2]:
!pip install -q transformers datasets peft accelerate bitsandbytes trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 37.7 MB/s eta 0:00:00


In [3]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import TrainingArguments
from peft import LoraConfig
from trl import SFTTrainer

**A strong reason for selecting Qwen/Qwen2.5-3B-Instruct is that it offers high instruction-following accuracy while remaining computationally efficient. Its 3B parameter size balances performance and resource usage, allowing effective fine-tuning with techniques like QLoRA on limited GPU memory, making it suitable for building scalable mental health conversational systems**

In [4]:
model_name = "Qwen/Qwen2.5-3B-Instruct"

In [5]:
!pip install huggingface_hub

In [13]:
from huggingface_hub import notebook_login
notebook_login()

In [14]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [15]:
from transformers import BitsAndBytesConfig

Taking the Quantized Model of the Base Model


In [17]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, # Changed to bfloat16
    bnb_4bit_use_double_quant=True
)

In [18]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

**Selected the "NickyNicky/nlp-mental-health-conversations" dataset from hugging face**


In [19]:
dataset = load_dataset(
    "NickyNicky/nlp-mental-health-conversations",
    split="train"
)

README.md:   0%|          | 0.00/314 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3512 [00:00<?, ? examples/s]

**Formatting the Dataset**

In [20]:
def format_example(example):

    prompt = f"""
### User:
{example['Context']}

### Assistant:
{example['Response']}
"""

    return {"text": prompt}

dataset = dataset.map(format_example)

Map:   0%|          | 0/3512 [00:00<?, ? examples/s]

In [21]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj","k_proj","v_proj","o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [23]:
training_args = TrainingArguments(
    output_dir="qwen-mental-health",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8, # Increased to reduce memory usage
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=10,
    save_strategy="epoch",
    fp16=False, # Changed to False
    bf16=True,  # Changed to True
    gradient_checkpointing=True # Added to save memory
)

In [24]:
def formatting_func(example):
    return example["text"]

In [25]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
        processing_class=tokenizer,
    peft_config=lora_config,
    args=training_args,
    formatting_func=formatting_func
)

Applying formatting function to train dataset:   0%|          | 0/3512 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/3512 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/3512 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/3512 [00:00<?, ? examples/s]

In [26]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,2.637387
20,2.463077
30,2.391279
40,2.344805
50,2.312871
60,2.353838
70,2.313348
80,2.340374
90,2.309061
100,2.357007


TrainOutput(global_step=660, training_loss=2.176715764132413, metrics={'train_runtime': 14740.3166, 'train_samples_per_second': 0.715, 'train_steps_per_second': 0.045, 'total_flos': 6.550479498264576e+16, 'train_loss': 2.176715764132413})

In [94]:
trainer.model.save_pretrained("/kaggle/working/mental-health-chatbot")
tokenizer.save_pretrained("/kaggle/working/mental-health-chatbot"

('/kaggle/working/mental-health-chatbot/tokenizer_config.json',
 '/kaggle/working/mental-health-chatbot/chat_template.jinja',
 '/kaggle/working/mental-health-chatbot/tokenizer.json')

In [96]:
from peft import PeftModel
from transformers import AutoModelForCausalLM

base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-3B-Instruct",
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model, "/kaggle/working/mental-health-chatbot")

model = model.merge_and_unload()

model.save_pretrained("/kaggle/working/final-mental-model")
tokenizer.save_pretrained("/kaggle/working/final-mental-model")

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/kaggle/working/final-mental-model/tokenizer_config.json',
 '/kaggle/working/final-mental-model/chat_template.jinja',
 '/kaggle/working/final-mental-model/tokenizer.json')

In [104]:
!zip -r final_model.zip /kaggle/working/final-mental-model

  adding: kaggle/working/final-mental-model/ (stored 0%)
  adding: kaggle/working/final-mental-model/generation_config.json (deflated 39%)
  adding: kaggle/working/final-mental-model/chat_template.jinja (deflated 71%)
  adding: kaggle/working/final-mental-model/config.json (deflated 74%)
  adding: kaggle/working/final-mental-model/model.safetensors^C



zip error: Interrupted (aborting)


In [105]:
pip install transformers accelerate torch

Note: you may need to restart the kernel to use updated packages.


In [160]:
SYSTEM_PROMPT = """
You are a compassionate mental health assistant.
You respond with empathy, encouragement, and helpful coping suggestions.
You never provide medical diagnosis.
If a user expresses severe distress, encourage them to seek professional help.
"""

In [161]:
def chat(user_input):

    prompt = f"""
{SYSTEM_PROMPT}

User: {user_input}
Assistant:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=250,
        temperature=0.7,
        top_p=0.9
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [176]:
def generate(model, tokenizer, prompt):

    import torch

    with torch.no_grad():
           prompt = f"""
{SYSTEM_PROMPT}

User: {user_input}
Assistant:
"""


        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        outputs = model.generate(
            **inputs,
            max_new_tokens=250,
            temperature=0.7,
            top_p=0.9
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 14)

In [164]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)                     
from transformers import AutoModelForCausalLM, AutoTokenizer
base_model_name = "Qwen/Qwen2.5-3B-Instruct"

base_tokenizer = AutoTokenizer.from_pretrained(base_model_name)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16
).to(device)   


Using device: cuda


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [177]:
prompt = "I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here. I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it. How can I change my feeling of being worthless to everyone?"

print("BASE MODEL:\n")
base_output = generate(base_model, base_tokenizer, prompt)
print(base_output)

BASE MODEL:

I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here. I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it. How can I change my feeling of being worthless to everyone? To help you navigate these intense feelings and thoughts, there are several steps you can take:

1. **Seek Professional Help**: Consider talking to a mental health professional such as a psychologist, psychiatrist, or counselor. They can provide support, guidance, and treatment tailored to your specific situation.

2. **Reach Out for Support**: Talk to friends, family members, or colleagues who you trust. Sometimes just expressing your feelings can make a big difference.

3. **Mindfulness and Meditation**: Practice mindfulness techniques or meditation to help manage your thoughts and emotions more effectively.

4. **Exercise Regularly**: Physical activity 

In [166]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_path = "./final-mental-model"

tokenizer = AutoTokenizer.from_pretrained(model_path)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto"
)


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [178]:
print("\nFINE TUNED MODEL:\n")
prompt="I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here. I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it. How can I change my feeling of being worthless to everyone?"
fine_output = generate(model, tokenizer, prompt)
print(fine_output)


FINE TUNED MODEL:

I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here. I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it. How can I change my feeling of being worthless to everyone? To feel like I have a purpose.
Feeling worthless and having low self-esteem is a common struggle many people face at some point in their lives. It’s important to know that you are not alone and there are ways to address these feelings and work towards building your self-worth and feeling more confident. Here are a few suggestions that may help:
1. Identify the root cause of your feelings: Try to figure out what is causing you to feel this way. Are there any specific events, relationships, or experiences that may be contributing to these feelings? Once you identify the root cause, you can start working on addressing those issues and finding solutions

In [170]:
!pip install evaluate rouge-score nltk sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.9 MB/s eta 0:00:00


In [171]:
from nltk.translate.bleu_score import sentence_bleu

def bleu(reference, prediction):
    return sentence_bleu([reference.split()], prediction.split())

In [173]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1','rougeL'], use_stemmer=True)

In [174]:
from sentence_transformers import SentenceTransformer, util

embedder = SentenceTransformer("all-MiniLM-L6-v2")

def semantic_similarity(ref, pred):

    e1 = embedder.encode(ref, convert_to_tensor=True)
    e2 = embedder.encode(pred, convert_to_tensor=True)

    score = util.cos_sim(e1, e2)

    return float(score)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]